
# Sequence 1 (no obstacle): stereo robot localisation and tracking,
using techniques from the course exercises and lectures:

- Image Processing lecture: background subtraction (temporal median),
  thresholding, morphology and connected components
- Week 2: Lucas-Kanade sparse optical flow
- Week 3: Stereo block matching / disparity
- Week 7: Standard matrix Kalman filter

Same structure and outputs as the Sequence 3 script, so the results of all
sequences look the same in the report.


In [1]:
import glob
import re
import csv
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import plotly.graph_objects as go


# 1. Parameters

In [2]:
LEFT_DIR = "1_no_obstacle/1_no_obstacle/left"
RIGHT_DIR = "1_no_obstacle/1_no_obstacle/right"
CALIB_FILE = "stereo_calib.npz"
OUTPUT_DIR = Path("robot_estimation_results_seq1")

# 0.001 if the stereo calibration translation is in millimetres;
# 1.0 if it is already in metres.
CALIB_LENGTH_TO_METRES = 0.001

FPS = 10.0

# Background subtraction (Image Processing lecture)
BACKGROUND_STEP = 4          # use every 4th frame for the temporal median
DIFF_THRESHOLD = 25          # grey-level change that counts as foreground
OPEN_SIZE = 3                # opening: removes single noisy pixels
CLOSE_SIZE = 15              # closing: fills holes in the dark robot body
MAST_SIZE = 25               # opening: removes the thin mast on top
MIN_BLOB_AREA = 800          # smallest blob (px) that can be the robot

# Stereo block matching (block-matching exercise, tuned for full resolution)
NUM_DISPARITIES = 96
BLOCK_SIZE = 15
MIN_VALID_DISPARITY = 0.0
MIN_VALID_PIXELS = 200       # valid disparity pixels needed on the robot mask

# Kalman filter noise
ACCEL_NOISE = 0.5            # m/s^2, small random acceleration (constant-speed robot)
SIGMA_PIXEL = 3.0            # centroid noise (px)
SIGMA_DISPARITY = 0.25       # disparity noise (px)
BORDER_SCALE = 25.0          # robot cut by the image border: trust it 25x less
MAX_MISSES = 15              # end the track after this many frames without a measurement

# Lucas-Kanade parameters (Week 2 exercise)
LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30, 0.01))

# Plotly axes, same as the other sequences:
# X: -10 to +10 m (left is negative), depth: 0 to 20 m, height: -2 to 10 m.
AXIS_HALF_RANGE = 10.0
AXIS_MAX = 10.0
MAX_CLOUD_POINTS = 25000
GRID_STEP = 1.0              # 1 m x 1 m grid in both maps

# Camera pose correction, same values as the other sequences:
# rotation about the X axis for the downward pitch, and a shift in Y for the height.
PITCH_ROT_X_DEG = 5.0
TRANSLATION_Y = -1.0


# 2. Load image names and stereo calibration

In [3]:
def natural_key(path):
    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", str(path))
    ]


def image_files(folder):
    files = []
    for extension in ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif"):
        files.extend(glob.glob(str(Path(folder) / extension)))
    return sorted(files, key=natural_key)


left_files = image_files(LEFT_DIR)
right_files = image_files(RIGHT_DIR)

if not left_files or len(left_files) != len(right_files):
    raise RuntimeError(
        f"Expected matching stereo images: {len(left_files)} left, "
        f"{len(right_files)} right."
    )

if len(left_files) < 2:
    raise RuntimeError("At least two stereo pairs are required.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(CALIB_FILE) as calib:
    mtxL, distL = calib["mtxL"], calib["distL"]
    mtxR, distR = calib["mtxR"], calib["distR"]
    R1, R2 = calib["R1"], calib["R2"]
    P1, P2, Q = calib["P1"], calib["P2"], calib["Q"]
    W, H = map(int, calib["image_size"])

mapL1, mapL2 = cv2.initUndistortRectifyMap(
    mtxL, distL, R1, P1, (W, H), cv2.CV_32FC1
)
mapR1, mapR2 = cv2.initUndistortRectifyMap(
    mtxR, distR, R2, P2, (W, H), cv2.CV_32FC1
)

# Focal length (px) and baseline (m) from Q, used for the depth noise
FOCAL_PX = Q[2, 3]
BASELINE_M = CALIB_LENGTH_TO_METRES / Q[3, 2]


def load_pair(index):
    left = cv2.imread(left_files[index])
    right = cv2.imread(right_files[index])

    if left is None or right is None:
        raise RuntimeError(f"Could not read stereo pair {index}.")

    if left.shape[1::-1] != (W, H) or right.shape[1::-1] != (W, H):
        raise RuntimeError(
            f"Image size must match calibration size {(W, H)}."
        )

    left = cv2.remap(left, mapL1, mapL2, cv2.INTER_LINEAR)
    right = cv2.remap(right, mapR1, mapR2, cv2.INTER_LINEAR)
    return left, right

RuntimeError: Expected matching stereo images: 0 left, 0 right.

# 3. Robot detection: background subtraction + morphology +
    connected components (Image Processing lecture)

In [ ]:
# Background model: per-pixel temporal MEDIAN of every 4th frame.
# The cameras are static and the robot covers each pixel for only a few
# frames, so the median is the empty hall.
print("Building the background model...")
background_frames = [
    load_pair(i)[0] for i in range(0, len(left_files), BACKGROUND_STEP)
]
background = np.median(np.array(background_frames), axis=0).astype(np.uint8)
del background_frames
cv2.imwrite(str(OUTPUT_DIR / "background.png"), background)


def foreground_mask(image):
    """Difference to the background -> threshold -> morphology."""
    difference = cv2.absdiff(image, background).max(axis=2)
    mask = (difference > DIFF_THRESHOLD).astype(np.uint8) * 255
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, np.ones((OPEN_SIZE, OPEN_SIZE), np.uint8)
    )
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE, np.ones((CLOSE_SIZE, CLOSE_SIZE), np.uint8)
    )
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (MAST_SIZE, MAST_SIZE))
    )
    return mask


def detect_robot(image):
    """The robot is the largest connected component of the foreground mask.

    Returns centre (u, v), bounding box (x, y, w, h), the robot's pixel mask
    and whether the box touches the left or right image border.
    """
    mask = foreground_mask(image)
    count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

    if count < 2:                      # label 0 is the background
        return None, None, None, False

    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    x, y, box_w, box_h, area = stats[k]

    if area < MIN_BLOB_AREA:
        return None, None, None, False

    border = bool(x <= 1 or x + box_w >= W - 1)
    bbox = (int(x), int(y), int(box_w), int(box_h))
    return centroids[k].astype(float), bbox, labels == k, border


Building the background model...


# 4. Stereo disparity and 3D position (block-matching exercise)

In [ ]:
stereo = cv2.StereoBM_create(numDisparities=NUM_DISPARITIES, blockSize=BLOCK_SIZE)
stereo.setMinDisparity(0)
stereo.setDisp12MaxDiff(1)
stereo.setUniquenessRatio(10)
stereo.setSpeckleRange(2)
stereo.setSpeckleWindowSize(100)


def disparity_map(left, right):
    gray_left = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
    return stereo.compute(gray_left, gray_right).astype(np.float32) / 16.0


def estimate_position(left, right, center, robot_mask):
    """3D position (m): mask centroid + MEDIAN disparity inside the robot mask."""
    disparity = disparity_map(left, right)

    values = disparity[robot_mask]
    values = values[np.isfinite(values) & (values > MIN_VALID_DISPARITY)]

    # Not enough disparity on the robot, e.g. in the left 96 columns,
    # which the right camera cannot see
    if len(values) < MIN_VALID_PIXELS:
        return None, disparity, None

    d = float(np.median(values))

    # Reproject pixel position and disparity through calibrated Q.
    cx, cy = center
    X, Y, Z, scale = Q @ np.array([cx, cy, d, 1.0])

    if abs(scale) < 1e-10:
        return None, disparity, d

    xyz = np.array([X, Y, Z]) / scale
    xyz *= CALIB_LENGTH_TO_METRES

    if not np.isfinite(xyz).all() or xyz[2] <= 0:
        return None, disparity, d

    return xyz, disparity, d

# 5. Standard matrix Kalman filter (Week 7)

In [ ]:
def measurement_noise(xyz, border):
    """R for one 3D measurement. Depth error grows with Z^2 (from Z = f*B/d)."""
    Z = xyz[2]
    sigma_z = Z**2 / (FOCAL_PX * BASELINE_M) * SIGMA_DISPARITY
    sigma_xy = SIGMA_PIXEL * Z / FOCAL_PX
    R = np.diag([sigma_xy**2, sigma_xy**2, sigma_z**2])
    return R * (BORDER_SCALE if border else 1.0)


def kalman_filter(measurements, dt, border):
    """
    Constant-velocity Kalman filter.
    State: [x, vx, y, vy, z, vz]
    Measurement: [x, y, z]
    Frames without a measurement are predicted only. The track ends after
    MAX_MISSES frames in a row without a measurement.
    Returns positions, velocities and a flag for prediction-only frames.
    """
    count = len(measurements)
    filtered = np.full((count, 3), np.nan)
    velocity = np.full((count, 3), np.nan)
    predicted_only = np.zeros(count, dtype=bool)

    valid = np.isfinite(measurements).all(axis=1)
    valid_indices = np.where(valid)[0]

    if len(valid_indices) == 0:
        return filtered, velocity, predicted_only

    F = np.array([
        [1, dt, 0,  0,  0,  0],
        [0,  1, 0,  0,  0,  0],
        [0,  0, 1, dt,  0,  0],
        [0,  0, 0,  1,  0,  0],
        [0,  0, 0,  0,  1, dt],
        [0,  0, 0,  0,  0,  1]
    ], dtype=float)

    Hm = np.array([
        [1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0]
    ], dtype=float)

    # Process noise: small random acceleration, one 2x2 block per axis
    G = np.array([[0.5 * dt**2], [dt]])
    Qm = np.kron(np.eye(3), G @ G.T) * ACCEL_NOISE**2
    I = np.eye(6)

    first = valid_indices[0]
    state = np.array([
        measurements[first, 0], 0,
        measurements[first, 1], 0,
        measurements[first, 2], 0
    ], dtype=float).reshape(6, 1)

    # Position: as uncertain as the first measurement; velocity: unknown (2 m/s std)
    covariance = np.eye(6) * 2.0**2
    covariance[np.ix_([0, 2, 4], [0, 2, 4])] = measurement_noise(
        measurements[first], border[first]
    )

    misses = 0
    for i in range(first, count):
        if i > first:
            state = F @ state
            covariance = F @ covariance @ F.T + Qm

        if valid[i]:
            Rm = measurement_noise(measurements[i], border[i])
            z = measurements[i].reshape(3, 1)
            error = z - Hm @ state
            S = Hm @ covariance @ Hm.T + Rm
            K = covariance @ Hm.T @ np.linalg.inv(S)

            state = state + K @ error
            covariance = (I - K @ Hm) @ covariance
            misses = 0
        else:
            misses += 1
            predicted_only[i] = True
            if misses > MAX_MISSES:
                break

        filtered[i] = state[[0, 2, 4], 0]
        velocity[i] = state[[1, 3, 5], 0]

    # Drop the predictions after the last measurement (robot has left the image)
    last = valid_indices[-1]
    filtered[last + 1:] = np.nan
    velocity[last + 1:] = np.nan
    predicted_only[last + 1:] = False

    return filtered, velocity, predicted_only

# 6. Process all stereo pairs

In [ ]:
n_frames = len(left_files)
uv_measurements = np.full((n_frames, 2), np.nan)
xyz_measurements = np.full((n_frames, 3), np.nan)
disparities = np.full(n_frames, np.nan)
boxes = [None] * n_frames
border_flags = np.zeros(n_frames, dtype=bool)

for i in range(n_frames):
    left, right = load_pair(i)

    center, bbox, robot_mask, border = detect_robot(left)

    if center is not None:
        uv_measurements[i] = center
        boxes[i] = bbox
        border_flags[i] = border

        xyz, disp, d = estimate_position(left, right, center, robot_mask)

        if d is not None:
            disparities[i] = d

        if xyz is not None:
            xyz_measurements[i] = xyz

    if (i + 1) % 25 == 0 or i == n_frames - 1:
        print(f"Processed {i + 1}/{n_frames} frames")

valid_uv = np.isfinite(uv_measurements).all(axis=1)
valid_xyz = np.isfinite(xyz_measurements).all(axis=1)

if valid_uv.sum() < 3:
    raise RuntimeError(
        "Too few detections. Check DIFF_THRESHOLD and MIN_BLOB_AREA."
    )

if valid_xyz.sum() < 3:
    raise RuntimeError(
        "Too few valid 3D measurements. Check disparity and calibration."
    )

detected_frames = np.where(valid_uv)[0]
xyz_frames = np.where(valid_xyz)[0]
print(f"2D detections: {valid_uv.sum()}/{n_frames} "
      f"(frames {detected_frames[0]}-{detected_frames[-1]})")
print(f"3D measurements: {valid_xyz.sum()}/{n_frames} "
      f"(frames {xyz_frames[0]}-{xyz_frames[-1]})")

xyz_filtered, velocity, predicted_only = kalman_filter(
    xyz_measurements, 1.0 / FPS, border_flags
)

# Constant speed / constant distance check, on frames with full-quality measurements
# (computed before the pose correction, in the camera frame)
good = valid_xyz & ~border_flags & np.isfinite(xyz_filtered).all(axis=1)
speed = np.linalg.norm(velocity[good], axis=1)           # m/s
depth = xyz_filtered[good, 2]                            # m

# --- Apply the same rotation and translation to the trajectory ---
theta = np.radians(PITCH_ROT_X_DEG)
R_x = np.array([
    [1, 0, 0],
    [0, np.cos(theta), -np.sin(theta)],
    [0, np.sin(theta), np.cos(theta)]
])
xyz_filtered = (xyz_filtered @ R_x.T)
xyz_filtered[:, 1] += TRANSLATION_Y
xyz_measurements_corrected = xyz_measurements @ R_x.T
xyz_measurements_corrected[:, 1] += TRANSLATION_Y
# -----------------------------------------------------------------

valid_3d_indices = np.where(
    np.isfinite(xyz_filtered).all(axis=1)
)[0]
first_3d = int(valid_3d_indices[0])
last_3d = int(valid_3d_indices[-1])

trajectory = xyz_filtered[first_3d:last_3d + 1]

Processed 25/256 frames
Processed 50/256 frames
Processed 75/256 frames
Processed 100/256 frames
Processed 125/256 frames
Processed 150/256 frames
Processed 175/256 frames
Processed 200/256 frames
Processed 225/256 frames
Processed 250/256 frames
Processed 256/256 frames
2D detections: 152/256 (frames 66-217)
3D measurements: 141/256 (frames 77-217)


# 7. Lucas-Kanade optical flow video (Week 2), with the detection


In [ ]:
video_path = OUTPUT_DIR / "optical_flow.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(str(video_path), fourcc, FPS, (W, H))

print("Generating optical flow video...")

left1, _ = load_pair(0)

for i in range(n_frames - 1):
    left2, _ = load_pair(i + 1)

    flow_image = left2.copy()

    # If the robot is detected in both consecutive frames, compute optical flow
    if valid_uv[i] and valid_uv[i + 1]:
        gray1 = cv2.cvtColor(left1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(left2, cv2.COLOR_BGR2GRAY)

        # Detect corners inside the robot's box in frame i
        x, y, box_w, box_h = boxes[i]
        x0, x1 = max(0, x), min(W, x + box_w)
        y0, y1 = max(0, y), min(H, y + box_h)

        roi = gray1[y0:y1, x0:x1]
        corners = cv2.goodFeaturesToTrack(
            roi, maxCorners=100, qualityLevel=0.005, minDistance=1
        )

        if corners is not None and len(corners) > 0:
            corners[:, 0, 0] += x0
            corners[:, 0, 1] += y0

            new_points, status, error = cv2.calcOpticalFlowPyrLK(
                gray1, gray2, corners, None, **LK_PARAMS
            )

            if new_points is not None:
                good_old = corners[status.ravel() == 1]
                good_new = new_points[status.ravel() == 1]

                for old, new in zip(good_old, good_new):
                    p1 = tuple(np.round(old.flatten()).astype(int))
                    p2 = tuple(np.round(new.flatten()).astype(int))

                    if np.linalg.norm(np.array(p2) - np.array(p1)) > 1:
                        cv2.line(flow_image, p1, p2, (0, 255, 0), 2)
                        cv2.circle(flow_image, p2, 3, (0, 0, 255), -1)

    # Detection box: green, or orange when the robot is cut by the image border
    if boxes[i + 1] is not None:
        x, y, box_w, box_h = boxes[i + 1]
        colour = (0, 165, 255) if border_flags[i + 1] else (0, 255, 0)
        cv2.rectangle(flow_image, (x, y), (x + box_w, y + box_h), colour, 2)

    text = f"frame {i + 1}"
    if np.isfinite(xyz_measurements[i + 1]).all():
        X, _, Z = xyz_measurements[i + 1]
        text += f"   X = {X:+.2f} m   Z = {Z:.2f} m"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
    cv2.rectangle(flow_image, (5, 5), (20 + tw, 20 + th), (0, 0, 0), -1)
    cv2.putText(flow_image, text, (12, 12 + th), cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (255, 255, 255), 2, cv2.LINE_AA)

    video_writer.write(flow_image)
    left1 = left2

video_writer.release()

Generating optical flow video...


# 8. 2D trajectory plot (top-down, 1 m grid)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# Keep the original camera/calibration coordinate convention.
ax.plot(
    trajectory[:, 0], trajectory[:, 2],
    linewidth=2, label="Kalman-filtered trajectory"
)
ax.scatter(
    xyz_measurements_corrected[valid_xyz, 0],
    xyz_measurements_corrected[valid_xyz, 2],
    s=12, alpha=0.4, label="Raw measurements"
)
ax.scatter(
    trajectory[0, 0], trajectory[0, 2],
    s=60, label="Start"
)
ax.scatter(
    trajectory[-1, 0], trajectory[-1, 2],
    s=60, label="End"
)
ax.scatter(0, 0, marker="^", s=80, color="black", label="Left camera")

ax.set_title("Sequence 1: top-down robot trajectory")
ax.set_xlabel("X (m)")
ax.set_ylabel("Camera depth (m)")
ax.xaxis.set_major_locator(MultipleLocator(GRID_STEP))
ax.yaxis.set_major_locator(MultipleLocator(GRID_STEP))
ax.grid(True)
ax.axis("equal")
ax.legend(loc="lower left")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "trajectory_2d.png", dpi=200)
plt.close(fig)

# 9. 3D point cloud and trajectory (20 m per axis)


In [ ]:
# The point cloud comes from frame 0, before the robot enters, so the
# robot does not appear in the static map.
cloud_left, cloud_right = load_pair(0)
cloud_disparity = disparity_map(cloud_left, cloud_right)

points = cv2.reprojectImageTo3D(cloud_disparity, Q)
points *= CALIB_LENGTH_TO_METRES

# --- Rotation about X + translation in Y ---
shape = points.shape
flat_points = points.reshape(-1, 3)

# Safely suppress warnings from NaN values in invalid disparity pixels
with np.errstate(invalid='ignore'):
    flat_points = flat_points @ R_x.T

points = flat_points.reshape(shape)
points[:, :, 1] += TRANSLATION_Y
# ---------------------------------------------------

cloud_mask = (
    (cloud_disparity > MIN_VALID_DISPARITY)
    & np.isfinite(points).all(axis=2)
    & (points[:, :, 2] > 0)
    & (-points[:, :, 1] >= 0)  # Filters out any remaining points below height 0
)

cloud = points[cloud_mask]
colors = cv2.cvtColor(cloud_left, cv2.COLOR_BGR2RGB)[cloud_mask]

if len(cloud) > MAX_CLOUD_POINTS:
    indices = np.random.default_rng(42).choice(
        len(cloud), MAX_CLOUD_POINTS, replace=False
    )
    cloud = cloud[indices]
    colors = colors[indices]

color_values = [f"rgb({r},{g},{b})" for r, g, b in colors]

fig3d = go.Figure()

if len(cloud) > 0:
    fig3d.add_trace(go.Scatter3d(
        x=cloud[:, 0],
        y=cloud[:, 2],
        z=-cloud[:, 1],
        mode="markers",
        marker=dict(size=1.5, color=color_values, opacity=0.55),
        name="Stereo point cloud"
    ))

fig3d.add_trace(go.Scatter3d(
    x=trajectory[:, 0],
    y=trajectory[:, 2],
    z=-trajectory[:, 1],
    mode="lines+markers",
    line=dict(width=6),
    marker=dict(size=3),
    name="Robot trajectory"
))

fig3d.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers",
    marker=dict(size=5, symbol="diamond"),
    name="Left camera origin"
))

fig3d.update_layout(
    title="Sequence 1: 3D stereo map and robot trajectory",
    scene=dict(
        xaxis=dict(title="X (m)",
                   range=[-AXIS_HALF_RANGE, AXIS_HALF_RANGE], dtick=GRID_STEP),
        yaxis=dict(title="Camera depth (m)",
                   range=[0, 2 * AXIS_MAX], dtick=GRID_STEP),
        zaxis=dict(title="Height / vertical coordinate -Y (m)",
                   range=[-2, AXIS_MAX], dtick=GRID_STEP),
        aspectmode="cube"
    ),
    margin=dict(l=0, r=0, b=0, t=45)
)

fig3d.write_html(OUTPUT_DIR / "trajectory_3d.html")


# 10. Save the track and print the summary


In [ ]:
with open(OUTPUT_DIR / "track.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["frame", "detected", "border", "u", "v", "disparity_px",
                     "X_m", "Y_m", "Z_m", "kf_X_m", "kf_Y_m", "kf_Z_m",
                     "prediction_only"])
    for i in range(n_frames):
        values = [*uv_measurements[i], disparities[i],
                  *xyz_measurements[i], *xyz_filtered[i]]
        writer.writerow([i, int(valid_uv[i]), int(border_flags[i])]
                        + ["" if not np.isfinite(v) else round(float(v), 4)
                           for v in values]
                        + [int(predicted_only[i])])

print("\nProcessing completed.")
print(f"Results saved in: {OUTPUT_DIR.resolve()}")
print(" - background.png")
print(" - optical_flow.mp4")
print(" - trajectory_2d.png")
print(" - trajectory_3d.html")
print(" - track.csv")

print(f"\nSpeed (Kalman, non-border frames): {speed.mean():.2f} ± {speed.std():.2f} m/s "
      f"= {speed.mean() / FPS * 1000:.1f} mm/frame")
print(f"Depth (Kalman, non-border frames): {depth.mean():.2f} ± {depth.std():.2f} m")

print(
    "\nThe 3D viewer axes each span 20 metres. "
    "The displayed depth axis is 0 to 20 m; X and height are -10 to +10 m."
)


Processing completed.
Results saved in: C:\Users\Jana Aly\pfas\final project\robot_estimation_results_seq1
 - background.png
 - optical_flow.mp4
 - trajectory_2d.png
 - trajectory_3d.html
 - track.csv

Speed (Kalman, non-border frames): 0.64 ± 0.09 m/s = 64.0 mm/frame
Depth (Kalman, non-border frames): 4.06 ± 0.10 m

The 3D viewer axes each span 20 metres. The displayed depth axis is 0 to 20 m; X and height are -10 to +10 m.
